# Kaggle Image → Video — LTX-Video 2B Distilled

This notebook **actually runs the model inside Kaggle** and writes a real `.mp4` to `/kaggle/working/outputs`.

Before running:
1. Kaggle → **Settings → Accelerator → GPU T4 x2**
2. Turn **Internet ON**
3. Run cells top-to-bottom.

Why LTX 2B here: it is much smaller and more realistic for Kaggle's T4 memory/RAM than the 34+ GB HunyuanVideo-1.5 Diffusers checkpoint.

The first model download can take a while. Later generations in the same session are much faster because the weights stay cached.

In [ ]:
# 1) Check the Kaggle GPU
import os, sys, torch

assert torch.cuda.is_available(), "No CUDA GPU found. In Kaggle Settings, set Accelerator = GPU T4 x2."

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory/1024**3:.1f} GB")

# Use one T4. LTX 2B fits on one with CPU offload; the second T4 is not required.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
# 2) Install official LTX-Video code.
# Put Hugging Face weights in Kaggle's temporary scratch space instead of /kaggle/working.
import os, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/tmp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/hf/transformers"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

repo = Path("/kaggle/working/LTX-Video")

if not repo.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Lightricks/LTX-Video.git",
        str(repo)
    ])

subprocess.check_call(
    f'{sys.executable} -m pip install -q -e ".[inference]"',
    cwd=str(repo),
    shell=True
)

print("Installed LTX-Video.")


In [ ]:
# 3) T4 compatibility patch + Kaggle-friendly config
#
# LTX's official 2B config uses BF16. Tesla T4 is a pre-Ampere GPU, so this notebook
# switches the inference path to FP16. We also disable the optional prompt-enhancer
# models so Kaggle does NOT download an extra multi-billion-parameter LLM.

from pathlib import Path
import re

repo = Path("/kaggle/working/LTX-Video")
src_cfg = repo / "configs/ltxv-2b-0.9.8-distilled.yaml"
t4_cfg = repo / "configs/ltxv-2b-0.9.8-distilled-t4.yaml"

cfg = src_cfg.read_text()
cfg = cfg.replace('precision: "bfloat16"', 'precision: "float16"')
cfg = re.sub(
    r'prompt_enhancement_words_threshold:\s*\d+',
    'prompt_enhancement_words_threshold: 0',
    cfg
)
t4_cfg.write_text(cfg)

inference_py = repo / "ltx_video/inference.py"
code = inference_py.read_text()

# Add an FP16 transformer branch if the upstream file does not already have one.
if 'precision == "float16"' not in code:
    pattern = (
        r'(elif precision == "bfloat16":\n'
        r'\s+return Transformer3DModel\.from_pretrained\(ckpt_path\)'
        r'\.to\(torch\.bfloat16\))'
    )
    replacement = (
        r'\1\n'
        r'    elif precision == "float16":\n'
        r'        return Transformer3DModel.from_pretrained(ckpt_path).to(torch.float16)'
    )
    code, n = re.subn(pattern, replacement, code)
    if n == 0:
        raise RuntimeError("Upstream LTX inference code changed; FP16 patch location was not found.")

# Keep VAE + text encoder on a dtype T4 supports well.
code = code.replace("vae = vae.to(torch.bfloat16)", "vae = vae.to(torch.float16)")
code = code.replace("text_encoder = text_encoder.to(torch.bfloat16)", "text_encoder = text_encoder.to(torch.float16)")
inference_py.write_text(code)

print("T4 config:", t4_cfg)
print("FP16 patch ready.")


## Launch the web app

Run the next cell after the setup cells finish.

It launches a **separate Gradio webpage**. Kaggle will print a public `https://....gradio.live` link; tap that link to open the generator in its own page.

Inside the page you can:
- upload a starting image
- type a motion prompt
- choose portrait / landscape / square
- set duration-ish frame count and seed
- click **Generate video**
- watch/download the resulting MP4

Keep the Kaggle notebook tab open while you use the Gradio page, because the Kaggle GPU is doing the generation.

In [ ]:
# 4) Launch a separate Gradio image-to-video web app

import os, sys, gc, time, uuid, shutil
from pathlib import Path

import torch
from PIL import Image, ImageOps

# Install Gradio if the Kaggle image does not already have a recent version.
try:
    import gradio as gr
except Exception:
    !pip install -q -U gradio
    import gradio as gr

sys.path.insert(0, "/kaggle/working/LTX-Video")
os.chdir("/kaggle/working/LTX-Video")

from ltx_video.inference import infer, InferenceConfig

OUTPUT_DIR = Path("/kaggle/working/outputs")
UPLOAD_DIR = Path("/kaggle/working/gradio_inputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_CONFIG = "/kaggle/working/LTX-Video/configs/ltxv-2b-0.9.8-distilled-t4.yaml"

DEFAULT_PROMPT = """The person in the reference image stays recognizable and naturally consistent.
They smile warmly, blink, subtly shift their posture, and move naturally.
Their hair and clothing move slightly with the motion.
Gentle handheld camera movement, photorealistic candid video, coherent anatomy,
smooth continuous motion, same face, same clothing, same room and lighting, no scene change."""

DEFAULT_NEGATIVE = """worst quality, blurry, jittery, inconsistent motion, distorted face,
deformed hands, extra fingers, duplicate person, morphing identity,
warped body, text, logo, watermark, scene cut"""

RES_PRESETS = {
    "Portrait 384×640 (recommended)": (640, 384),
    "Landscape 640×384": (384, 640),
    "Square 512×512": (512, 512),
    "Portrait 448×768 (heavier)": (768, 448),
    "Landscape 768×448 (heavier)": (448, 768),
}

FRAME_OPTIONS = {
    "2.7 sec / 65 frames": 65,
    "4.0 sec / 97 frames": 97,
    "5.0 sec / 121 frames": 121,
}

def _latest_video_after(start_time: float):
    vids = [
        p for p in OUTPUT_DIR.glob("*.mp4")
        if p.stat().st_mtime >= start_time - 1
    ]
    if not vids:
        vids = list(OUTPUT_DIR.glob("*.mp4"))
    if not vids:
        raise RuntimeError("Generation finished but no MP4 was found.")
    return max(vids, key=lambda p: p.stat().st_mtime)

def generate_video(image, prompt, negative_prompt, resolution, duration, seed):
    if image is None:
        raise gr.Error("Upload a starting image first.")
    if not prompt or not prompt.strip():
        raise gr.Error("Enter a motion prompt.")

    # Gradio passes a PIL image here.
    img = ImageOps.exif_transpose(image).convert("RGB")
    uid = uuid.uuid4().hex[:10]
    image_path = UPLOAD_DIR / f"input_{uid}.png"
    img.save(image_path)

    height, width = RES_PRESETS[resolution]
    num_frames = FRAME_OPTIONS[duration]
    seed = int(seed)

    gc.collect()
    torch.cuda.empty_cache()

    start = time.time()

    config = InferenceConfig(
        pipeline_config=PIPELINE_CONFIG,
        prompt=prompt.strip(),
        negative_prompt=(negative_prompt or "").strip(),
        conditioning_media_paths=[str(image_path)],
        conditioning_strengths=[1.0],
        conditioning_start_frames=[0],
        height=height,
        width=width,
        num_frames=num_frames,
        frame_rate=24,
        seed=seed,
        offload_to_cpu=True,
        output_path=str(OUTPUT_DIR),
    )

    infer(config)

    generated = _latest_video_after(start)

    # Give the result a stable Gradio-friendly filename.
    final_path = OUTPUT_DIR / f"video_{uid}.mp4"
    if generated != final_path:
        shutil.copy2(generated, final_path)

    return str(final_path), f"Done in {(time.time()-start)/60:.1f} min · {width}×{height} · {num_frames} frames · seed {seed}"

with gr.Blocks(title="Kaggle LTX Image → Video") as demo:
    gr.Markdown(
        "# 🎬 Kaggle Image → Video\n"
        "Upload an image, describe the motion, then generate an actual MP4 on the Kaggle GPU."
    )

    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type="pil", label="Starting image")
            prompt_in = gr.Textbox(
                value=DEFAULT_PROMPT,
                label="Motion prompt",
                lines=8,
            )
            negative_in = gr.Textbox(
                value=DEFAULT_NEGATIVE,
                label="Negative prompt",
                lines=4,
            )

            with gr.Row():
                resolution_in = gr.Dropdown(
                    list(RES_PRESETS.keys()),
                    value="Portrait 384×640 (recommended)",
                    label="Resolution",
                )
                duration_in = gr.Dropdown(
                    list(FRAME_OPTIONS.keys()),
                    value="2.7 sec / 65 frames",
                    label="Length",
                )

            seed_in = gr.Number(value=42, precision=0, label="Seed")
            generate_btn = gr.Button("Generate video", variant="primary")

        with gr.Column():
            video_out = gr.Video(label="Generated video")
            status_out = gr.Textbox(label="Status", interactive=False)

    generate_btn.click(
        fn=generate_video,
        inputs=[
            image_in,
            prompt_in,
            negative_in,
            resolution_in,
            duration_in,
            seed_in,
        ],
        outputs=[video_out, status_out],
    )

# share=True creates the separate public gradio.live page.
# debug=True keeps Kaggle output visible if an inference error occurs.
demo.queue(default_concurrency_limit=1).launch(
    share=True,
    debug=True,
    show_error=True,
)
